# Module 3 Coding Assignment: Exploring Transformer Architecture with GPT-2


This assignment explores the transformer architecture using a decoder-only model (GPT-2). We will load a pre-trained GPT-2 model, generate text, and delve into its architecture.

# Instructions
The assignment is broken down into the following steps: installing required libraries, loading the model and tokenizer, exploring the model's architecture, generating a single token using the model, and finally, setting up a text-generation pipeline for generating text from a given prompt.

By completing this assignment, you will gain a deeper understanding of how transformer models like GPT-2 function and their capabilities in natural language processing.


## Step 1. Setup
* Install Required Libraries
* Set Hugging Face Token - You will need to add your hugging face token to colab secrets. Visit: https://huggingface.co/settings/tokens to set up your hugging face token, we are reading the LLama2 model only so you can set up a read only token. Then paste your token to your Colab Secret and name it "HF_TOKEN". See details here: https://x.com/GoogleColab/status/1719798406195867814


In [ ]:
!pip install transformers --quiet

## Step 2. Load the Model and Tokenizer
* Load the gpt2 model and its tokenizer. (This is the smallest version of GPT-2, with 124M parameters.)
* Wrap the model and tokenizer in a text-generation pipeline.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
# Load the GPT-2 model and tokenizer
# You will need to set a HF_TOKEN in your secrets

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Device set to use cpu


## Step 3. Exploring the Model's Architecture
* Print the model object (model) to view its overall architecture.
* Access specific components like `model.transformer.h[0]` to examine individual blocks within the transformer.

In [ ]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

You can also access a specific block to see its structure.

In [ ]:
model.transformer.h[0]

GPT2Block(
  (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (attn): GPT2Attention(
    (c_attn): Conv1D(nf=2304, nx=768)
    (c_proj): Conv1D(nf=768, nx=768)
    (attn_dropout): Dropout(p=0.1, inplace=False)
    (resid_dropout): Dropout(p=0.1, inplace=False)
  )
  (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (mlp): GPT2MLP(
    (c_fc): Conv1D(nf=3072, nx=768)
    (c_proj): Conv1D(nf=768, nx=3072)
    (act): NewGELUActivation()
    (dropout): Dropout(p=0.1, inplace=False)
  )
)

## Step 4. Generating a Single Token
* Tokenize the input prompt and convert it to token IDs.
* Pass the token IDs through the transformer blocks.
* Get the output before the language model head.
* Pass the output through the language model head to get logits.
* The logits represent the probability distribution over the vocabulary.
* Extract the logits for the last token in the sequence.
* Use `argmax` to find the most likely next token.
* Decode the token ID to get the actual token.



In [ ]:
# prompt
prompt = "the sky is blue and the grass is"

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
print(input_ids)

tensor([[1169, 6766,  318, 4171,  290,  262, 8701,  318]])


In [ ]:
# Pass token IDs through the transformer blocks
model_output = model.transformer(input_ids)
print(model_output.last_hidden_state.shape)

torch.Size([1, 8, 768])


In [ ]:
# Pass output through the language model head
lm_head_output = model.lm_head(model_output.last_hidden_state)
print(lm_head_output.shape)

torch.Size([1, 8, 50257])


In [ ]:
# Get the logits for the last token
last_token_logits = lm_head_output[0, -1, :]

# Find the token with the highest probability
next_token_id = last_token_logits.argmax(-1)
print(next_token_id)

tensor(4077)


In [ ]:
# Decode the token ID
next_token = tokenizer.decode(next_token_id)
print(next_token)

 green


GPT-2 correctly guessed the next word which is green.

## Step 5. Set up a pipeline and test it with a simple prompt

* Create a text-generation pipeline using pipeline() from the transformers library, specifying the model, tokenizer, and generation parameters.
* Define a prompt for text generation.
* Use the pipeline to generate text based on the prompt (generator(prompt)).
* Print the generated text to observe the model's output.

In [ ]:
# Create a text-generation pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=True,  # Include the prompt in the output
    max_new_tokens=30,       # Generate up to 30 tokens
    do_sample=False          # Allow randomness in text generation
)

In [ ]:
# Define a prompt
prompt = "Hello, I'm a language model, "

# Generate text
output = generator(prompt)

# Print the generated text
print(output[0]['generated_text'])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Hello, I'm a language model,  and I'm not a programmer. I'm a programmer. I'm a programmer. I'm a programmer. I'm a programmer. I'm


### What do you think about the model?
The output generated by GPT-2, where it repeats "I'm a programmer" after identifying itself as a language model, reveals an interesting quirk. It appears the model gets stuck in a loop, highlighting the limitations of early LLMs in maintaining context and generating coherent, consistent text.

This behavior is typical for models like GPT-2, which, while impressive, can sometimes exhibit unexpected or repetitive patterns. As we progress to more advanced models like GPT-3 and GPT-4, these issues become less prevalent due to improvements in architecture and training data. These newer models are better at understanding and maintaining context, resulting in more logical and nuanced text generation.


# Summary
This assignment provided hands-on experience with the GPT-2 transformer model. We explored the model's architecture, generated text by predicting individual tokens, and utilized a text-generation pipeline for creating longer sequences. These exercises demonstrated the power and flexibility of transformer models in natural language processing tasks. By understanding the underlying principles and practical application of GPT-2, you have gained valuable insights into the world of large language models and their capabilities.
